In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

# RPY2 Imports (Modern Style)
import rpy2.robjects as robjects
from rpy2.robjects import pandas2ri
from rpy2.robjects.conversion import localconverter

# Threshold Determination and Optimal Species Selection

In [ ]:
# gut_analysis.py
import os
import numpy as np
import pandas as pd
import pyreadr              # pip install pyreadr
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import normalize

# ----------------------
# 1) Load RData
# ----------------------
rdata_path = "Gut_profile_all.RData"   # change if needed
result = pyreadr.read_r(rdata_path)    # returns OrderedDict of all objects

# Check available objects
print("Objects in RData:", list(result.keys()))

# Retrieve objects
# pyreadr returns pandas objects for data.frames; for matrices they may come as DataFrame
Gut_Metadata = result.get('Gut_Metadata')
Gut_SpeciesProfile = result.get('Gut_SpeciesProfile')

if Gut_Metadata is None or Gut_SpeciesProfile is None:
    raise ValueError("Gut_Metadata or Gut_SpeciesProfile not found in RData. Available: "
                     + ", ".join(result.keys()))

print("Gut_Metadata shape:", Gut_Metadata.shape)
print("Gut_SpeciesProfile shape:", Gut_SpeciesProfile.shape)

# Quick peek
print(Gut_Metadata.head())
print(Gut_SpeciesProfile.iloc[:5, :5])

In [ ]:
import pandas as pd
import numpy as np

# -----------------------------
# 1. Setup: replicate R objects
# -----------------------------
species_profile = Gut_SpeciesProfile.copy()
metadata = Gut_Metadata.copy()

# Ensure indices align (R uses rownames → pandas index)
species_profile.index = species_profile.index.astype(str)
metadata.index = metadata.index.astype(str)

# Add study_name column from metadata
species_profile["study_name"] = metadata["study_name"]

# Get list of study names
study_list = species_profile["study_name"].unique().tolist()


# --------------------------------------------------
# 2. Define detection-frequency function (Python)
# --------------------------------------------------
def compute_detection(df, var1_list, grouping_variable, grouping_list):
    """
    df: pandas DataFrame containing species + grouping variable column
    var1_list: list of species names (columns) to compute detection for
    grouping_variable: column name used for grouping (e.g., "study_name")
    grouping_list: list of group values (unique values of that column)

    Returns:
        detection_df (rows = species, columns = grouping levels)
    """
    detection_matrix = pd.DataFrame(
        np.zeros((len(var1_list), len(grouping_list))),
        index=var1_list,
        columns=grouping_list
    )

    for var1 in var1_list:
        for group in grouping_list:
            group_data = df[df[grouping_variable] == group][var1]

            # numerator: how many samples have non-zero abundance
            nonzero_count = (group_data != 0).sum()

            # denominator: total samples in that group
            total_count = group_data.shape[0]

            # detection value
            detection_matrix.loc[var1, group] = nonzero_count / total_count if total_count > 0 else np.nan

    return detection_matrix


In [ ]:
import pandas as pd
import numpy as np

# Copy df
df = species_profile.copy()

# Extract study column
study = df["study_name"]

# Remove study_name from species data
species_only = df.drop(columns=["study_name"])

# Boolean presence/absence (True = detected)
# NOTE: ensure NaN is treated as NOT detected (matches R behavior)
presence = species_only.notna() & (species_only != 0)

# Group sizes (number of samples in each study) — matches R denominator
group_counts = study.value_counts().sort_index()

# sum of detections within each study (groupby by study values)
det_sum = presence.groupby(study).sum()  # DataFrame: index=study, columns=species

# detection frequency: divide counts by group sizes (aligned on index)
AllSpeciesDetectionPattern = det_sum.div(group_counts, axis=0)

print(AllSpeciesDetectionPattern)

In [ ]:
import numpy as np
import pandas as pd

# -------------------------
# Preconditions: the following must exist:
# - species_profile : pandas DataFrame (samples x species) and a 'study_name' column
# - AllSpeciesDetectionPattern : pandas DataFrame (either species x studies or studies x species)
# - study_list : list-like of study names (original R order, if you want to preserve it)
# If study_list isn't available, we'll derive it from species_profile['study_name'].
# -------------------------
if "study_list" not in globals():
    study_list = species_profile["study_name"].astype(str).unique().tolist()

# -------------------------
# Ensure AllSpeciesDetectionPattern orientation:
# We want species as rows (index) and studies as columns.
# Use name-overlap heuristics rather than fragile index.equals checks.
# -------------------------
# If AllSpeciesDetectionPattern rows overlap study names and columns overlap species names,
# then it is transposed (studies x species) and needs transposing.
det = AllSpeciesDetectionPattern.copy()

rows = set(det.index.astype(str))
cols = set(det.columns.astype(str))
species_from_mean = None

# compute AllSpeciesMeanAbundance (species x studies) like R does, if not already present
grouped = species_profile.drop(columns=["study_name"]).groupby(species_profile["study_name"]).mean()
AllSpeciesMeanAbundance = grouped.T  # rows = species, cols = studies

# Use names to decide whether to transpose det
species_names = set(AllSpeciesMeanAbundance.index.astype(str))
study_names = set(AllSpeciesMeanAbundance.columns.astype(str))

# If det rows look like studies and det columns look like species -> transpose
if len(rows & study_names) > len(rows & species_names) and len(cols & species_names) > len(cols & study_names):
    det = det.T
    # recompute sets after transpose
    rows = set(det.index.astype(str))
    cols = set(det.columns.astype(str))

# Final sanity: ensure species overlap
if len(rows & species_names) == 0:
    raise RuntimeError("After attempting orientation fixes, detection rows (det.index) still have no overlap with species names from mean-abundance table. Inspect names/indexes.")

# Now det: species x studies
AllSpeciesDetectionPattern = det.copy()

# -------------------------
# Threshold grids (match R: seq(0.05, 0.95, by = 0.05))
# -------------------------
study_thresholds = np.round(np.arange(0.05, 1.0, 0.05), 10)      # length 19
detection_thresholds = np.round(np.arange(0.05, 1.0, 0.05), 10)   # length 19

n_det = len(detection_thresholds)
n_study_thr = len(study_thresholds)

# Initialize matrices (shape n_det x n_study_thr) same as your R matrices
df_numb_species = np.full((n_det, n_study_thr), np.nan, dtype=float)
df_representation_90_plus = np.full((n_det, n_study_thr), np.nan, dtype=float)
df_representation_70_minus = np.full((n_det, n_study_thr), np.nan, dtype=float)

# Helpful items
det_df = AllSpeciesDetectionPattern  # species x studies DataFrame
n_studies = det_df.shape[1]
species_index = det_df.index.astype(str).tolist()
study_list_local = det_df.columns.astype(str).tolist()

# For denominators in representation (%) the R code used length(study_list) (original study_list)
# We'll keep that behavior; fall back to number of studies present if study_list is missing
denom_studies = len(study_list) if (study_list is not None and len(study_list) > 0) else n_studies

# Convert detection values to numpy for speed
det_values = det_df.values  # shape: n_species x n_studies

# -------------------------
# Main loops: outer detection thresholds, inner study thresholds (same order as your R code)
# -------------------------
for i, detect in enumerate(detection_thresholds):
    # For each species compute how many studies have detection >= detect
    count_studies_ge = (det_values >= detect).sum(axis=1)   # length n_species
    prop_studies_ge = count_studies_ge / float(n_studies)

    for j, study_perc in enumerate(study_thresholds):
        mask_species = prop_studies_ge >= study_perc
        df_numb_species[i, j] = int(mask_species.sum())

        temp_species = np.array(species_index)[mask_species].tolist()

        # Keep only species present in AllSpeciesMeanAbundance (defensive)
        temp_species_in_mean = [s for s in temp_species if s in AllSpeciesMeanAbundance.index]

        if len(temp_species_in_mean) == 0:
            # No species: match R behaviour -> set 0
            df_representation_90_plus[i, j] = 0.0
            df_representation_70_minus[i, j] = 0.0
            continue

        # subset mean-abundance to selected species
        subset_mean = AllSpeciesMeanAbundance.loc[temp_species_in_mean, :]

        # Align studies: use intersection of columns between subset_mean and det_df
        common_studies = [s for s in subset_mean.columns if s in det_df.columns]
        if len(common_studies) == 0:
            # no common studies -> set zero
            df_representation_90_plus[i, j] = 0.0
            df_representation_70_minus[i, j] = 0.0
            continue

        col_sums = subset_mean[common_studies].sum(axis=0)  # per-study sums

        # Use original study_list length for denominator to replicate R's behavior
        df_representation_90_plus[i, j] = float((col_sums >= 0.90).sum()) / float(denom_studies)
        df_representation_70_minus[i, j] = float((col_sums < 0.70).sum()) / float(denom_studies)

# -------------------------
# Build final DataFrame matching R's as.numeric() (column-major flatten)
# -------------------------
vec_numb = df_numb_species.flatten(order='F')
vec_rep90 = df_representation_90_plus.flatten(order='F')
vec_rep70 = df_representation_70_minus.flatten(order='F')

df_gut_associated_identification = pd.DataFrame({
    "number_of_species": vec_numb.astype(float),
    "representation_90_plus": vec_rep90.astype(float),
    "representation_70_minus": vec_rep70.astype(float)
})

# Build matching threshold labels (column-major flattening)
# R's loop order: for (i in detection_thresholds) for (j in study_thresholds) df[i,j]=...
# Flattening column-major means detection_threshold repeats for each column (study_threshold).
det_list = np.tile(detection_thresholds, n_study_thr)     # detection varying fastest within each column
study_list_rep = np.repeat(study_thresholds, n_det)       # study threshold repeated n_det times per column

df_gut_associated_identification["detection_threshold"] = det_list
df_gut_associated_identification["study_threshold"] = study_list_rep

# -------------------------
# OPTIONAL: also return the raw matrices with row/col labels (handy for heatmaps)
df_numb_species_df = pd.DataFrame(df_numb_species, index=detection_thresholds, columns=study_thresholds)
df_rep90_df = pd.DataFrame(df_representation_90_plus, index=detection_thresholds, columns=study_thresholds)
df_rep70_df = pd.DataFrame(df_representation_70_minus, index=detection_thresholds, columns=study_thresholds)

# -------------------------
print("Done. Example head:")
print(df_gut_associated_identification.head())

import os

# Define output folder name
output_folder = "Gut_Identification_Results"

# Create the folder if it doesn't exist
os.makedirs(output_folder, exist_ok=True)
print(f"Output folder ready: {output_folder}")

# Save CSVs into the folder
df_gut_associated_identification.to_csv(os.path.join(output_folder, "df_gut_associated_identification.csv"), index=False)
df_numb_species_df.to_csv(os.path.join(output_folder, "df_numb_species_matrix.csv"))
df_rep90_df.to_csv(os.path.join(output_folder, "df_representation_90_plus.csv"))
df_rep70_df.to_csv(os.path.join(output_folder, "df_representation_70_minus.csv"))

print(f"All files saved successfully in '{output_folder}'")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 7))

# Chartreuse2 ≈ #7FFF00   | coral4 ≈ #8B3E2F
sns.scatterplot(
    data=df_gut_associated_identification,
    x="number_of_species",
    y="representation_90_plus",
    s=50,
    color="#7FFF00",
    label="representation_90_plus"
)

sns.scatterplot(
    data=df_gut_associated_identification,
    x="number_of_species",
    y="representation_70_minus",
    s=50,
    color="#8B3E2F",
    label="representation_70_minus"
)

plt.xlabel("number_of_species", fontsize=15)
plt.ylabel("representation", fontsize=15)
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)
plt.grid(True, linewidth=0.8, color="gray")    # like R panel.grid.major
#plt.legend(fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df90 = df_rep90_df.copy()

# Build annotation matrix from number_of_species column
annotation_matrix = np.array(df_gut_associated_identification["number_of_species"]).reshape(19, 19)

# Convert to integers (fixes the fmt="d" error)
annotation_matrix = annotation_matrix.astype(int)

# Transpose like R: t(annotation_matrix)
annotation_matrix_t = annotation_matrix.T

plt.figure(figsize=(12, 10))
sns.heatmap(
    df90,
    annot=annotation_matrix_t,
    fmt="d",                   # now works because annotations are ints
    cmap="viridis",
    cbar=True,
    linewidths=0.5
)

plt.xlabel("study_threshold", fontsize=14)
plt.ylabel("detection_threshold", fontsize=14)
plt.title("representation_90_plus (numbers overlaid)", fontsize=16)
plt.tight_layout()
plt.show()


In [ ]:
subset_df = df_gut_associated_identification[
    (df_gut_associated_identification["representation_90_plus"] >= 0.90) &
    (df_gut_associated_identification["representation_70_minus"] <= 0.70)
]

subset_df


# Extraction of the optimal species

In [ ]:
import numpy as np
import pandas as pd

# ============================================================
# Load Data (already assumed loaded in your Python workspace)
# Gut_SpeciesProfile, Gut_Metadata
# ============================================================

species_profile = Gut_SpeciesProfile.copy()
metadata = Gut_Metadata.copy()

# Add study_name column
species_profile["study_name"] = metadata["study_name"].astype(str)

# unique study list
study_list = species_profile["study_name"].unique().tolist()


In [ ]:
import pandas as pd
import numpy as np

# Copy df
df = species_profile.copy()

# Extract study column
study = df["study_name"].astype(str)

# Remove study_name from species data
species_only = df.drop(columns=["study_name"])

# Boolean presence/absence (True = detected)
# NOTE: ensure NaN is treated as NOT detected (matches R behavior)
presence = species_only.notna() & (species_only != 0)

# Group sizes (denominator)
group_counts = study.value_counts().sort_index()

# Sum of detections within each study (numerator)
det_sum = presence.groupby(study).sum()   # shape: study x species

# detection frequency matrix (study x species)
AllSpeciesDetectionPattern = det_sum.div(group_counts, axis=0)

# Transpose to species x studies (R format)
AllSpeciesDetectionPattern = AllSpeciesDetectionPattern.T

print("Detection matrix shape:", AllSpeciesDetectionPattern.shape)

In [ ]:
study_thresholds = np.arange(0.05, 1.00, 0.05)
detection_thresholds = np.arange(0.05, 1.00, 0.05)

df_numb_species = np.zeros((len(study_thresholds), len(detection_thresholds)))

values = AllSpeciesDetectionPattern.values  # species × studies
n_studies = AllSpeciesDetectionPattern.shape[1]

for i, detect in enumerate(detection_thresholds):
    proportion = (values >= detect).mean(axis=1)  # species-level mean(x >= detect)

    for j, study_perc in enumerate(study_thresholds):
        df_numb_species[j, i] = (proportion >= study_perc).sum()

df_numb_species_df = pd.DataFrame(
    df_numb_species,
    index=study_thresholds,
    columns=detection_thresholds
)

In [ ]:
where_466 = np.argwhere(df_numb_species == 466)

if where_466.size == 0:
    raise ValueError("No threshold pair produced EXACTLY 820 species using fast detection matrix.")

row_i, col_i = where_466[0]

detect = detection_thresholds[col_i]
study_perc = study_thresholds[row_i]

print("Found threshold pair → detect:", detect, "study threshold:", study_perc)


In [ ]:
proportion = (values >= detect).mean(axis=1)

selected_species = AllSpeciesDetectionPattern.index[proportion >= study_perc].tolist()

print("Number of selected species:", len(selected_species))

In [ ]:
filtered_species_profile_466 = species_profile[selected_species]
filtered_metadata_466 = metadata.copy()

print(filtered_species_profile_466.shape)
print(filtered_metadata_466.shape)

# Analysis of REM Results over 466

In [ ]:
# FIX 1: Ensure lowercase 'v' matches your folder
py_dir_path = "REM_Results_466_v2" 
print(f"\n[2] Loading Python Data from: {py_dir_path}")

try:
    est_mat = pd.read_csv(os.path.join(py_dir_path, "rem_est_matrix.csv"), index_col=0)
    dir_mat = pd.read_csv(os.path.join(py_dir_path, "rem_dir_matrix.csv"), index_col=0)
    pval_mat = pd.read_csv(os.path.join(py_dir_path, "rem_pval_matrix.csv"), index_col=0)
    qval_mat = pd.read_csv(os.path.join(py_dir_path, "rem_qval_matrix.csv"), index_col=0)
    consistency_mat = pd.read_csv(os.path.join(py_dir_path, "rem_consistency_matrix.csv"), index_col=0)
    print(f"    [✔] Matrices loaded. Shape: {est_mat.shape}")

except FileNotFoundError:
    # FIX 2: Do not use exit(). It crashes the notebook kernel.
    print(f"    [✘] ERROR: Files not found in '{py_dir_path}'. Check spelling!")
    # Stop execution safely
    raise

In [ ]:
qval_mat.shape

In [ ]:
consistency_mat.shape

In [ ]:
print("Overall Min:", dir_mat.min().min())
print("Overall Max:", dir_mat.max().max())

In [ ]:
dir_mat.shape

# Creation of New Directionality Matrix from the REM Results

In [ ]:
import numpy as np
import pandas as pd
import os

# --- 1. Setup & Initialization ---
# Create a new DataFrame initialized with 0s, matching the shape (466x466) and index of your data
# This acts like: rem_combined_results$new_dir <- matrix(NA, ...) in R
# We use 0.0 as default so it handles floats; integers will cast automatically if needed.
new_dir_mat = pd.DataFrame(0.0, index=est_mat.index, columns=est_mat.columns)

# Pre-calculate the sign of the estimates (+1, -1, 0)
# R code: sign(est)
est_sign = np.sign(est_mat)

# --- 2. Apply Transformation Rules ---

# Rule 1: Strongest (qval <= 0.05) -> Score 3
# R code: rem_combined_results$new_dir[qval <= 0.05] <- 3 * sign(est[qval <= 0.05])
mask1 = qval_mat <= 0.05
new_dir_mat[mask1] = 3.0 * est_sign[mask1]

# Rule 2: Medium (qval > 0.05 AND pval <= 0.05) -> Score 2
# R code: condition2 <- (qval > 0.05) & (pval <= 0.05)
mask2 = (qval_mat > 0.05) & (pval_mat <= 0.05)
new_dir_mat[mask2] = 2.0 * est_sign[mask2]

# Rule 3: Weak (pval > 0.05 AND pval <= 0.1) -> Score 1
# R code: condition3 <- (pval > 0.05) & (pval <= 0.1)
mask3 = (pval_mat > 0.05) & (pval_mat <= 0.1)
new_dir_mat[mask3] = 1.0 * est_sign[mask3]

# Rule 4: Noise (pval > 0.1) -> Score 0
# R code: condition4 <- pval > 0.1
# Note: Since we initialized with 0.0, this is technically redundant but good for strict compliance.
mask4 = pval_mat > 0.1
new_dir_mat[mask4] = 0.0

# --- 3. Save the Result ---
# Saves to the same directory as your input files
output_filename = "rem_new_dir_matrix.csv"
output_path = os.path.join(py_dir_path, output_filename)

new_dir_mat.to_csv(output_path)

print(f"\n[Success] New Direction Matrix created with shape {new_dir_mat.shape}")
print(f"Saved to: {output_path}")

In [ ]:
new_dir_mat.max().max()

In [ ]:
# Optional: Preview the counts of values to verify logic worked
print("\nValue Counts in New Matrix:")
# Flatten the matrix to count occurrences of -3, -2, -1, 0, 1, 2, 3
print(new_dir_mat.stack().value_counts().sort_index())

print(new_dir_mat.shape)

# Preparing the Embedding Matrix

In [ ]:
import numpy as np
import pandas as pd
import os
from scipy.stats import rankdata

# --- 1. Setup ---
# Define the path (ensure this matches your previous step)
py_dir_path = "REM_Results_466_v2"

print(f"Loading data from: {py_dir_path}")

# Load the necessary matrices created in previous steps
# We need 'new_dir' (created in the last python step) and 'consistency' (loaded earlier)
try:
    new_dir_mat = pd.read_csv(os.path.join(py_dir_path, "rem_new_dir_matrix.csv"), index_col=0)
    consistency_mat = pd.read_csv(os.path.join(py_dir_path, "rem_consistency_matrix.csv"), index_col=0)
    print("Files loaded successfully.")
except FileNotFoundError:
    print("Error: Could not find input files. Run the previous step first.")
    raise

# Initialize Embedding Matrix (start with 0s)
embedding_df = pd.DataFrame(0.0, index=new_dir_mat.index, columns=new_dir_mat.columns)

# --- 2. Define Rank Scale Function ---
# R's rank() works on the flattened matrix, so we must flatten, rank, and reshape.
def get_ranked_consistency(cons_df):
    # Flatten to 1D array
    flat_values = cons_df.values.flatten()
    
    # Rank (method='average' matches default R behavior for ties)
    ranks = rankdata(flat_values, method='average')
    
    # Min-Max Scale to [0, 1]
    # y <- (rank(x) - min(rank(x)))/(max(rank(x)) - min(rank(x)))
    min_r = np.min(ranks)
    max_r = np.max(ranks)
    scaled = (ranks - min_r) / (max_r - min_r)
    
    # Handle NaNs (replace with 0)
    scaled = np.nan_to_num(scaled, nan=0.0)
    
    # Reshape back to original DataFrame shape
    return pd.DataFrame(scaled.reshape(cons_df.shape), 
                        index=cons_df.index, 
                        columns=cons_df.columns)

# Calculate the Ranked Consistency Matrix
ranked_consistency = get_ranked_consistency(consistency_mat)

# --- 3. Apply Update Rules ---

# Rule A: Consistency Filter
# We only want to update cells where Consistency > 0.7. 
# Everything else stays 0 (initialized value).
high_consistency_mask = consistency_mat > 0.7

# Rule B: new_dir == 2 (Medium Strength)
# Logic: Embedding = 0.25 * (1 + ranked_consistency)
mask_dir_2 = (new_dir_mat == 2) & high_consistency_mask
embedding_df[mask_dir_2] = 0.25 * (1 + ranked_consistency[mask_dir_2])

# Rule C: new_dir == 3 (High Strength)
# Logic: Embedding = 0.5 * (1 + ranked_consistency)
mask_dir_3 = (new_dir_mat == 3) & high_consistency_mask
embedding_df[mask_dir_3] = 0.5 * (1 + ranked_consistency[mask_dir_3])

# Rule D: Ensure Cleanups
# (Any NaNs or negative directions are already handled by initialization to 0 
# and the strict masking above, but we do a final safety fill)
embedding_df = embedding_df.fillna(0)

# --- 4. Save Output ---
output_filename = "rem_embedding_matrix.csv"
output_path = os.path.join(py_dir_path, output_filename)

embedding_df.to_csv(output_path)

print(f"\n[Success] Embedding Matrix created for {embedding_df.shape[0]} species.")
print(f"Saved to: {output_path}")

# Optional: Inspect non-zero values to verify it worked
print("\nSample of non-zero embedding values:")
print(embedding_df.stack().loc[lambda x: x > 0].head())

In [ ]:
print(embedding_df.min().min())

print(embedding_df.max().max())

# 466 PCoA Embedding using Eucledian

In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from sklearn.manifold import MDS
from sklearn.metrics import euclidean_distances
from matplotlib import rcParams

# --- 1. Load Data ---
py_dir_path = "REM_Results_466_v2"
embedding_path = os.path.join(py_dir_path, "rem_embedding_matrix.csv")

try:
    # Load the Embedding Matrix (466 Species x 466 Species)
    embedding_df = pd.read_csv(embedding_path, index_col=0)
    print(f"Loaded Embedding Matrix: {embedding_df.shape}")
except FileNotFoundError:
    print("Error: 'rem_embedding_matrix.csv' not found. Please run the previous embedding generation script.")
    # Stop execution if file is missing
    raise

# --- 2. Compute Distance Matrix ---
# Equivalent to R: dist(Embedding_df, method = "euclidean")
dist_matrix = euclidean_distances(embedding_df)

# --- 3. Perform PCoA (MDS) ---
# Equivalent to R: cmdscale(dist_matrix, k = 2)
mds = MDS(n_components=2, dissimilarity='precomputed', random_state=42, n_init=4, max_iter=300)
pcoa_results = mds.fit_transform(dist_matrix)

# Create DataFrame for plotting
pcoa_coords_df = pd.DataFrame(pcoa_results, columns=["PC1", "PC2"], index=embedding_df.index)

# --- 4. Plotting (Exact Replica of ggplot2 style) ---

# A. Configure Font to Times New Roman
rcParams['font.family'] = 'sans-serif'
#rcParams['font.serif'] = ['Times New Roman']

# B. Create Plot
plt.figure(figsize=(20, 10))
ax = plt.gca()

# C. The Points (Orange, Alpha 0.9)
# s=100 in matplotlib is roughly size=3 in ggplot
plt.scatter(pcoa_coords_df["PC1"], pcoa_coords_df["PC2"], 
            color="orange", alpha=0.9, s=100, edgecolors='none')

# D. Titles and Labels
plt.title("PCoA Embedding (466 Species)", fontsize=16, fontweight='bold', pad=20)
plt.xlabel("PC 1", fontsize=14)
plt.ylabel("PC 2", fontsize=14)

# E. Theme Customization (Minimal + Border)
# Turn on all spines to create the 'box' border effect
for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_edgecolor('black')
    spine.set_linewidth(1.5)

# Grid lines (Light gray)
ax.grid(True, color='gray', linestyle='-', linewidth=0.5, alpha=0.3)
ax.set_axisbelow(True) # Ensure grid is behind the points

# Tick label sizes
ax.tick_params(axis='both', which='major', labelsize=12)

# F. Show Plot (No Saving)
plt.tight_layout()
plt.show()

# 466 PCoA Embeddings using Bray Curtis

In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from sklearn.manifold import MDS
from sklearn.metrics import pairwise_distances
from matplotlib import rcParams

# --- 1. Load Data ---
py_dir_path = "REM_Results_466_v2"
embedding_path = os.path.join(py_dir_path, "rem_embedding_matrix.csv")

try:
    # Load the Embedding Matrix (466 Species x 466 Species)
    # Using a NEW variable name for Bray-Curtis session
    embedding_df_bc = pd.read_csv(embedding_path, index_col=0)
    print(f"Loaded Embedding Matrix for Bray-Curtis: {embedding_df_bc.shape}")
except FileNotFoundError:
    print("Error: 'rem_embedding_matrix.csv' not found.")
    raise

# --- 2. Compute Distance Matrix (Bray-Curtis) ---
# NOTE: Bray-Curtis requires non-negative data. 
# If your embeddings contain negative values (e.g. from PCA/SVD), 
# we shift them to be positive to make the metric valid.
if (embedding_df_bc.values < 0).any():
    print("⚠️ Notice: Embedding contains negative values. Shifting data to enable Bray-Curtis...")
    embedding_df_bc = embedding_df_bc - embedding_df_bc.min().min()

print("Computing Bray-Curtis Distance Matrix...")
# 'n_jobs=-1' ensures we use all CPU cores (Parallel processing)
dist_matrix_bc = pairwise_distances(embedding_df_bc, metric='braycurtis', n_jobs=-1)

# --- 3. Perform PCoA (MDS) ---
# Using the same MDS settings as your original code
mds_bc = MDS(n_components=2, dissimilarity='precomputed', random_state=42, n_init=4, max_iter=300)
pcoa_results_bc = mds_bc.fit_transform(dist_matrix_bc)

# Create DataFrame for plotting with new variable names
pcoa_coords_bc_df = pd.DataFrame(pcoa_results_bc, columns=["PC1", "PC2"], index=embedding_df_bc.index)

# --- 4. Plotting (Exact Replica of ggplot2 style) ---

# A. Configure Font
rcParams['font.family'] = 'sans-serif'

# B. Create Plot
plt.figure(figsize=(20, 10))
ax = plt.gca()

# C. The Points (using a distinct color 'teal' to differentiate from Orange Euclidean plot)
# You can change this back to 'orange' if you prefer identical styling.
plt.scatter(pcoa_coords_bc_df["PC1"], pcoa_coords_bc_df["PC2"], 
            color="teal", alpha=0.9, s=100, edgecolors='none')

# D. Titles and Labels
plt.title("PCoA Embedding (466 Species) - Bray Curtis", fontsize=16, fontweight='bold', pad=20)
plt.xlabel("PC 1", fontsize=14)
plt.ylabel("PC 2", fontsize=14)

# E. Theme Customization (Minimal + Border)
for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_edgecolor('black')
    spine.set_linewidth(1.5)

# Grid lines
ax.grid(True, color='gray', linestyle='-', linewidth=0.5, alpha=0.3)
ax.set_axisbelow(True)

# Tick label sizes
ax.tick_params(axis='both', which='major', labelsize=12)

# F. Show Plot
plt.tight_layout()
plt.show()

# Optional: Inspect the first few rows
print("\nFirst 5 rows of Bray-Curtis PCoA Coordinates:")
print(pcoa_coords_bc_df.head())

# Grid Label Analysis over the PCoA

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import MDS
from sklearn.metrics import euclidean_distances
from matplotlib import rcParams

# ==============================================================================
# PART 1: COMPUTE PCoA (MDS)
# ==============================================================================
print("Step 1: Computing PCoA Embedding...")

# 1. Compute Distance Matrix (Euclidean)
# Equivalent to R: dist(Embedding_df, method = "euclidean")
dist_matrix = euclidean_distances(embedding_df)

# 2. Perform PCoA (Using MDS with precomputed dissimilarity)
# Equivalent to R: cmdscale(dist_matrix, k = 2)
mds = MDS(n_components=2, dissimilarity='precomputed', random_state=42, 
          n_init=4, max_iter=300, normalized_stress=False)
pcoa_results = mds.fit_transform(dist_matrix)

# 3. Create Coordinates DataFrame
pcoa_coords_df = pd.DataFrame(pcoa_results, columns=["PC1", "PC2"], index=embedding_df.index)
pcoa_coords_df["Species"] = pcoa_coords_df.index

# ==============================================================================
# PART 2: QUARTILE GRID BINNING
# ==============================================================================
print("Step 2: Calculating Quartile Grids...")

# 1. Define Boundaries using Quantiles (0%, 25%, 50%, 75%, 100%)
# Equivalent to R: quantile(..., probs = seq(0, 1, 0.25))
x_breaks = pcoa_coords_df["PC1"].quantile([0, 0.25, 0.5, 0.75, 1.0]).values
y_breaks = pcoa_coords_df["PC2"].quantile([0, 0.25, 0.5, 0.75, 1.0]).values

# 2. Assign Each Species to a Grid Cell
# Equivalent to R: cut(..., include.lowest = TRUE)
# labels=False returns integers 0, 1, 2, 3. We add +1 to match R's 1-based indexing
pcoa_coords_df["Col_Bin"] = pd.cut(pcoa_coords_df["PC1"], bins=x_breaks, labels=False, include_lowest=True) + 1
pcoa_coords_df["Row_Bin"] = pd.cut(pcoa_coords_df["PC2"], bins=y_breaks, labels=False, include_lowest=True) + 1

# 3. Create Grid ID (G11 to G44)
# Note: Row is Y-axis, Col is X-axis
pcoa_coords_df["Grid_ID"] = "G" + pcoa_coords_df["Row_Bin"].astype(str) + pcoa_coords_df["Col_Bin"].astype(str)

# 4. Check Distribution (Should be balanced)
print("\nGrid Distribution Summary:")
print(pcoa_coords_df["Grid_ID"].value_counts().sort_index())

# ==============================================================================
# PART 3: PLOTTING (Times New Roman, Borders, Grid Labels)
# ==============================================================================
print("\nStep 3: Generating Plot...")

# A. Calculate Label Coordinates (Midpoints of variable bins)
# We calculate the center of the gap between quantiles
x_centers = [(x_breaks[i] + x_breaks[i+1])/2 for i in range(4)]
y_centers = [(y_breaks[i] + y_breaks[i+1])/2 for i in range(4)]

# B. Configure Font
rcParams['font.family'] = 'sans-serif'
#rcParams['font.serif'] = ['Times New Roman']

# C. Plot Setup
plt.figure(figsize=(20, 10))
ax = plt.gca()

# D. Plot Points
plt.scatter(pcoa_coords_df["PC1"], pcoa_coords_df["PC2"], 
            color="orange", alpha=0.9, s=80, edgecolors='none', zorder=2)

# E. Add Grid Lines (The Quartile Boundaries)
for x in x_breaks[1:-1]: # Skip min/max to avoid plotting on border
    plt.axvline(x, color="gray", linestyle="--", linewidth=0.8, alpha=0.7, zorder=1)
for y in y_breaks[1:-1]:
    plt.axhline(y, color="gray", linestyle="--", linewidth=0.8, alpha=0.7, zorder=1)

# F. Add Grid Labels (G11, G12...)
for r_idx in range(4):     # Rows 1-4
    for c_idx in range(4): # Cols 1-4
        label = f"G{r_idx+1}{c_idx+1}"
        plt.text(x_centers[c_idx], y_centers[r_idx], label, 
                 fontsize=14, fontweight='bold', ha='center', va='center', 
                 color='black', alpha=0.8, zorder=3)

# G. Styling (Borders, Titles)
plt.title("PCoA Embedding (Quartile Grid Analysis)", fontsize=16, fontweight='bold', pad=15)
plt.xlabel("PC 1", fontsize=14, fontweight='bold')
plt.ylabel("PC 2", fontsize=14, fontweight='bold')

# Create the black border box
for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_edgecolor('black')
    spine.set_linewidth(1.5)

# Ticks
ax.tick_params(axis='both', which='major', labelsize=12)

# Show Plot
plt.tight_layout()
plt.show()

# ==============================================================================
# PART 4: DATA EXTRACTION (Create Dictionary of Dataframes)
# ==============================================================================
print("Step 4: Extracting Abundance Profiles per Grid...")

Grid_Extraction_Results = {}
all_grids = sorted(pcoa_coords_df["Grid_ID"].unique())

for gid in all_grids:
    # A. Identify Species in this Grid
    species_in_this_grid = pcoa_coords_df[pcoa_coords_df["Grid_ID"] == gid].index.tolist()
    
    # B. Extract Species Profile (Columns = Species)
    # We intersect to ensure we only grab species that exist in the profile data
    valid_species = [sp for sp in species_in_this_grid if sp in filtered_species_profile_466.columns]
    
    if len(valid_species) > 0:
        grid_profile = filtered_species_profile_466[valid_species]
    else:
        grid_profile = pd.DataFrame() # Empty if no match
        print(f"[Warning] Grid {gid}: No matching species columns found.")

    # C. Store in Dictionary
    Grid_Extraction_Results[gid] = {
        "Grid_ID": gid,
        "Species_Count": len(valid_species),
        "Species_List": valid_species,
        "Abundance_Data": grid_profile
    }
    
    # Optional Progress Print
    print(f"Processed {gid}: Extracted profile for {len(valid_species)} species.")

# ==============================================================================
# PART 5: VERIFICATION
# ==============================================================================
print("\n--- Verification: G11 Content ---")
print(f"Species Count: {Grid_Extraction_Results['G11']['Species_Count']}")
print("First 5 Species in G11:")
print(Grid_Extraction_Results['G11']['Species_List'][:5])
print("\nAbundance Data Shape for G11:")
print(Grid_Extraction_Results['G11']['Abundance_Data'].shape)

# Applying RPCA for each Grid

In [ ]:
import os

# --- CRITICAL: Keep this to prevent thread thrashing ---
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

import numpy as np
import pandas as pd
import pickle 
from tqdm import tqdm
from joblib import Parallel, delayed, cpu_count

# ==============================================================================
# CLASS: Robust PCA 
# ==============================================================================
class R_PCA:
    def __init__(self, D, mu=None, lmbda=None):
        self.D = D
        self.S = np.zeros(self.D.shape)
        self.Y = np.zeros(self.D.shape)

        if mu:
            self.mu = mu
        else:
            self.mu = np.prod(self.D.shape) / (4 * np.sum(np.abs(self.D)))

        self.mu_inv = 1 / self.mu

        if lmbda:
            self.lmbda = lmbda
        else:
            self.lmbda = 1 / np.sqrt(np.max(self.D.shape))

    @staticmethod
    def shrinkage(x, tau):
        return np.sign(x) * np.maximum(np.abs(x) - tau, 0)

    @staticmethod
    def svd_threshold(x, tau):
        U, S, V = np.linalg.svd(x, full_matrices=False)
        return np.dot(U, np.dot(np.diag(R_PCA.shrinkage(S, tau)), V))

    def fit(self, tol=None, max_iter=1000):
        iter = 0
        err = np.inf 
        
        Sk = self.S
        Yk = self.Y
        Lk = np.zeros(self.D.shape)

        if tol:
            _tol = tol
        else:
            _tol = 1E-7 * np.linalg.norm(self.D)

        while (err > _tol) and (iter < max_iter):
            Lk = self.svd_threshold(
                self.D - Sk + self.mu_inv * Yk, self.mu_inv)
            Sk = self.shrinkage(
                self.D - Lk + self.mu_inv * Yk, self.mu_inv * self.lmbda)
            Yk = Yk + self.mu * (self.D - Lk - Sk)
            err = np.linalg.norm(self.D - Lk - Sk)
            iter += 1
            
        self.L = Lk
        self.S = Sk
        return Lk, Sk, err

# ==============================================================================
# WORKER FUNCTION
# ==============================================================================
def process_grid(args):
    gid, df = args
    if df.shape[1] < 2:
        return gid, {"Status": "Skipped (Not enough species)"}
    
    try:
        mat_abundance = df.values
        rpca = R_PCA(mat_abundance)
        L, S, error = rpca.fit(max_iter=1000)
        
        return gid, {
            "L_Matrix": pd.DataFrame(L, index=df.index, columns=df.columns),
            "S_Matrix": pd.DataFrame(S, index=df.index, columns=df.columns),
            "Convergence_Error": error
        }
    except Exception as e:
        return gid, {"Error": str(e)}

# ==============================================================================
# MAIN EXECUTION (JOBLIB VERSION)
# ==============================================================================
if __name__ == '__main__':
    
    # 1. Check CPU Count
    total_cores = cpu_count()
    use_cores = max(1, total_cores - 2) # Leave 2 cores free for OS/Jupyter
    
    print(f"Total CPUs detected: {total_cores}")
    print(f"Using: {use_cores} cores for RPCA.")

    # 2. Prepare Tasks
    if 'Grid_Extraction_Results' not in locals():
        raise ValueError("Grid_Extraction_Results dictionary not found.")

    tasks = []
    for gid, data in Grid_Extraction_Results.items():
        tasks.append((gid, data['Abundance_Data']))

    print(f"Starting Parallel RPCA on {len(tasks)} Grids...")

    # 3. Execute with Joblib
    # n_jobs=use_cores: Uses multiple cores
    # backend="loky": Robust backend for notebooks
    results_list = Parallel(n_jobs=use_cores, backend="loky")(
        delayed(process_grid)(task) for task in tqdm(tasks, desc="RPCA Progress")
    )

    # 4. Convert List back to Dictionary
    RPCA_Grid_Results = {gid: res for gid, res in results_list}

    # 5. Save Results
    output_file = "RPCA_Grid_Results_Parallel.pkl"
    with open(output_file, "wb") as f:
        pickle.dump(RPCA_Grid_Results, f)
        
    print(f"\n[Success] RPCA completed. Saved to: {output_file}")
    
    # Verification
    if "G11" in RPCA_Grid_Results:
        res = RPCA_Grid_Results["G11"]
        if "L_Matrix" in res:
            print(f"G11 L-Matrix Shape: {res['L_Matrix'].shape}")
        else:
            print(f"G11 Status: {res}")

In [ ]:
import pickle
import pandas as pd

# --- 1. Load the Pickle File ---
file_path = "RPCA_Grid_Results_Parallel.pkl"

print(f"Loading {file_path}...")
with open(file_path, "rb") as f:
    RPCA_Results = pickle.load(f)

print("[Success] Data loaded!")

# --- 2. Inspect the Structure ---
# The result is a Dictionary where Keys = Grid IDs (G11, G12...)
grids_found = sorted(RPCA_Results.keys())
print(f"\nTotal Grids Processed: {len(grids_found)}")
print(f"Grid IDs: {grids_found}")

# --- 3. How to Extract Data for ONE Grid (e.g., G11) ---
target_grid = "G11"

if target_grid in RPCA_Results:
    grid_data = RPCA_Results[target_grid]
    
    # Check if it was successful (contains matrices) or failed (contains error message)
    if "L_Matrix" in grid_data:
        L_Mat = grid_data["L_Matrix"]  # This is a Pandas DataFrame
        S_Mat = grid_data["S_Matrix"]  # This is a Pandas DataFrame
        Error = grid_data["Convergence_Error"]
        
        print(f"\n--- Data for {target_grid} ---")
        print(f"L-Matrix Shape: {L_Mat.shape}")
        print(f"S-Matrix Shape: {S_Mat.shape}")
        print(f"Convergence Error: {Error}")
        
        # Preview the data
        print("\nTop 5 rows of L-Matrix (Stable Background):")
        print(L_Mat.iloc[:5, :5]) # Show first 5 rows/cols
        
    else:
        print(f"\n[!] Grid {target_grid} did not return matrices.")
        print(f"Status Message: {grid_data}")
else:
    print(f"\n[!] Grid {target_grid} not found in results.")

# Robust PC1 Extraction & Metadata Correlation

In [ ]:
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from scipy.stats import spearmanr

# --- 1. Setup ---
# We compare the "Robust" signal (L-Matrix) vs "Original" signal (L + S)
target_vars = ["age_num", "BMI", "Glucose"]
results_list = []

print("Starting PC1 Analysis & Metadata Correlation...")

# --- 2. Iterate Through Grids ---
# We use the keys from your loaded pickle file
for gid in sorted(RPCA_Results.keys()):
    
    # A. Get Matrices
    # Robust Data = L Matrix
    df_robust = RPCA_Results[gid]["L_Matrix"]
    # Original Data = L + S (Mathematically, Original = LowRank + Sparse + Error)
    # This is the safest way to reconstruct original data without reloading files
    df_original = df_robust + RPCA_Results[gid]["S_Matrix"]
    
    # B. Compute PC1 (Principal Component 1)
    # We use scikit-learn PCA
    pca = PCA(n_components=1)
    
    # Fit on Original (Noisy)
    pc1_orig = pca.fit_transform(df_original).flatten()
    var_orig = pca.explained_variance_ratio_[0] * 100
    
    # Fit on Robust (Clean)
    pc1_rob = pca.fit_transform(df_robust).flatten()
    var_rob = pca.explained_variance_ratio_[0] * 100
    
    # Improvement in Variance Explained
    imp_var = var_rob - var_orig
    
    # --- 3. Correlate with Metadata ---
    # Align samples between PC1 and Metadata
    common_samples = df_robust.index.intersection(filtered_metadata_466.index)
    
    if len(common_samples) < 10:
        continue # Skip if too few matching patients
        
    # Subset Data
    p_orig_sub = pd.Series(pc1_orig, index=df_original.index).loc[common_samples]
    p_rob_sub  = pd.Series(pc1_rob, index=df_robust.index).loc[common_samples]
    meta_sub   = filtered_metadata_466.loc[common_samples]
    
    for var in target_vars:
        if var not in meta_sub.columns: continue
            
        # Get metadata vector
        m_vec = pd.to_numeric(meta_sub[var], errors='coerce')
        
        # Remove NAs (valid pairs only)
        valid_mask = ~np.isnan(m_vec) & ~np.isnan(p_rob_sub)
        if valid_mask.sum() < 20: continue # Need at least 20 data points
        
        # Spearman Correlation
        # We use absolute value because sign (+/-) is arbitrary in PCA
        cor_o, _ = spearmanr(p_orig_sub[valid_mask], m_vec[valid_mask])
        cor_r, p_val = spearmanr(p_rob_sub[valid_mask], m_vec[valid_mask])
        
        cor_o = abs(cor_o)
        cor_r = abs(cor_r)
        
        # Save Result
        results_list.append({
            "Grid_ID": gid,
            "Meta_Var": var,
            "N_Samples": valid_mask.sum(),
            "Original_Cor": round(cor_o, 4),
            "Robust_Cor": round(cor_r, 4),
            "P_Value": p_val,
            "Improvement": "YES" if cor_r > cor_o else "NO",
            "PC1_Var_Gain": round(imp_var, 2)
        })

# --- 4. Create Summary Table ---
df_results = pd.DataFrame(results_list)

# Filter for Significant Discoveries (P < 0.05 AND Improvement = YES)
sig_discoveries = df_results[
    (df_results["P_Value"] < 0.05) & 
    (df_results["Improvement"] == "YES")
].sort_values("Robust_Cor", ascending=False)

print("\n=== TOP DISCOVERIES (Where RPCA beat Standard PCA) ===")
if not sig_discoveries.empty:
    # Display top 10 rows
    print(sig_discoveries[["Grid_ID", "Meta_Var", "Original_Cor", "Robust_Cor", "P_Value", "PC1_Var_Gain"]].head(15).to_string(index=False))
else:
    print("No significant improvements found.")

# Save
df_results.to_csv("RPCA_Metadata_Correlations.csv", index=False)
print("\n[Success] Full results saved to 'RPCA_Metadata_Correlations.csv'")

# Checking for the Variance

In [ ]:
import numpy as np
import pandas as pd
import pickle
from sklearn.decomposition import PCA

# --- 1. Load Data ---
# We need the pickle file containing L and S matrices
file_path = "RPCA_Grid_Results_Parallel.pkl"

if 'RPCA_Grid_Results' not in locals():
    print(f"Loading {file_path}...")
    with open(file_path, "rb") as f:
        RPCA_Grid_Results = pickle.load(f)
else:
    print("Using loaded RPCA_Grid_Results.")

# --- 2. Initialize Analysis ---
variance_summary = []
print("\nStarting PC1 Variance Extraction for all grids...")

# --- 3. Loop Through Grids ---
for gid in sorted(RPCA_Grid_Results.keys()):
    
    res = RPCA_Grid_Results[gid]
    
    # Check if grid has valid matrices (skip if error)
    if "L_Matrix" not in res:
        continue
        
    # A. Get Matrices
    df_robust = res["L_Matrix"]
    df_sparse = res["S_Matrix"]
    
    # Reconstruct Original (L + S)
    # This ensures we are comparing exactly the same data points
    df_original = df_robust + df_sparse
    
    # B. Perform PCA (SVD)
    pca = PCA(n_components=1)
    
    # 1. PCA on Original (Noisy)
    pca.fit(df_original)
    var_orig = pca.explained_variance_ratio_[0] * 100
    
    # 2. PCA on Robust (Clean L-Matrix)
    pca.fit(df_robust)
    var_rob = pca.explained_variance_ratio_[0] * 100
    
    # C. Calculate Improvement
    improvement = var_rob - var_orig
    
    # Store
    variance_summary.append({
        "Grid_ID": gid,
        "Original_Var_Explained": round(var_orig, 2),
        "Robust_Var_Explained": round(var_rob, 2),
        "Improvement": round(improvement, 2)
    })

# --- 4. Create and Print Report ---
df_variance = pd.DataFrame(variance_summary)

print("\n--- PC1 Variance Analysis (Original vs. Robust) ---")
print(df_variance.to_string(index=False))

# --- 5. Scientific Interpretation ---
avg_imp = df_variance["Improvement"].mean()
print(f"\nAverage Variance Improvement: {avg_imp:.2f}%")

if avg_imp > 0:
    print("Result: POSITIVE. The L-Matrix has a stronger, clearer dominant signal than the noisy original data.")
else:
    print("Result: NEUTRAL. The noise was actually part of the main signal.")

# Save for records
df_variance.to_csv("RPCA_Variance_Analysis.csv", index=False)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# --- 1. Prepare Data ---
# Load the results if not already in memory
try:
    df_variance = pd.read_csv("RPCA_Variance_Analysis.csv")
except FileNotFoundError:
    # Use existing dataframe if file not saved yet
    if 'df_variance' not in locals():
        raise ValueError("Please run the Variance Analysis script first.")

# Convert from Wide to Long format for Seaborn plotting
df_plot = df_variance.melt(
    id_vars="Grid_ID", 
    value_vars=["Original_Var_Explained", "Robust_Var_Explained"],
    var_name="Type", 
    value_name="Variance"
)

# Rename labels for the legend
df_plot["Type"] = df_plot["Type"].map({
    "Original_Var_Explained": "Original (Noisy)",
    "Robust_Var_Explained": "Robust (L-Matrix)"
})

# --- 2. Create Plot ---
plt.figure(figsize=(16, 8))
sns.set_style("white") # Clean background

# Draw Bar Chart
ax = sns.barplot(
    data=df_plot, 
    x="Grid_ID", 
    y="Variance", 
    hue="Type", 
    palette=["#95a5a6", "#2980b9"], # Gray for Original, Blue for Robust
    edgecolor="black",
    alpha=0.9
)

# --- 3. Add Annotations (The "Gain") ---
# We loop through the grids to add the "+X%" labels on top of the blue bars
for i, row in df_variance.iterrows():
    # Coordinates
    x_pos = i  # Barplot indices correspond to row numbers
    y_pos = row["Robust_Var_Explained"]
    gain = row["Improvement"]
    
    # Only label significant gains (>5%)
    if gain > 5:
        ax.text(
            x_pos + 0.2,   # Shift slightly right to sit over the Blue bar
            y_pos + 2,     # Shift slightly up
            f"+{gain:.0f}%", 
            ha='center', va='bottom', 
            fontsize=11, fontweight='bold', color='red'
        )

# --- 4. Styling (Publication Ready) ---
plt.title("RPCA Signal Improvement: Variance Explained by PC1", fontsize=18, fontweight='bold', pad=20)
plt.ylabel("Variance Explained (%)", fontsize=14, fontweight='bold')
plt.xlabel("Grid ID", fontsize=14, fontweight='bold')
plt.ylim(0, 115) # Leave room for labels
plt.legend(title="Data Source", fontsize=12, loc="upper right")

# Add standard border
for spine in ax.spines.values():
    spine.set_edgecolor('black')
    spine.set_linewidth(1.5)

plt.grid(axis='y', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

# Separation of 3 Data Frames

In [ ]:
import pandas as pd
import numpy as np
import pickle

# --- 1. Load Data ---
file_path_pkl = "RPCA_Grid_Results_Parallel.pkl"
# Assuming 'Grid_Extraction_Results' and 'filtered_metadata_466' are already in memory.
# If not, load them:
if 'Grid_Extraction_Results' not in locals():
    # You would typically load this from a pickle or reconstruct it, but 
    # based on previous steps, it should be available.
    raise ValueError("Grid_Extraction_Results is missing. Please run the Grid Extraction step.")

if 'filtered_metadata_466' not in locals():
    # Load metadata if needed
    # filtered_metadata_466 = pd.read_csv("metadata.csv", index_col=0) 
    pass 

if 'RPCA_Grid_Results' not in locals():
    print(f"Loading {file_path_pkl}...")
    with open(file_path_pkl, "rb") as f:
        RPCA_Grid_Results = pickle.load(f)

print("Initializing Volume Dataframes...")

# --- 2. Setup Master Dataframes ---
# Get Master Sample List (from metadata to ensure alignment)
master_samples = filtered_metadata_466.index.tolist()
grid_ids = sorted(RPCA_Grid_Results.keys())

# Initialize empty DataFrames with NaNs
# Rows = All Samples, Cols = 16 Grids
df_vol_orig = pd.DataFrame(np.nan, index=master_samples, columns=grid_ids)
df_vol_rob  = pd.DataFrame(np.nan, index=master_samples, columns=grid_ids)

# --- 3. Calculate Volumes (Row Sums) ---
print("Calculating Grid Volumes...")

for gid in grid_ids:
    
    # Check if grid exists in both datasets
    if gid not in Grid_Extraction_Results or gid not in RPCA_Grid_Results:
        print(f"Skipping {gid} (Missing from data)")
        continue
        
    # A. Get Matrices
    # Original Data (Abundance Profile)
    df_orig = Grid_Extraction_Results[gid]['Abundance_Data']
    
    # Robust Data (L-Matrix)
    res = RPCA_Grid_Results[gid]
    if "L_Matrix" not in res:
        print(f"Skipping {gid} (RPCA failed)")
        continue
    df_rob = res["L_Matrix"]
    
    # B. Identify Matching Samples
    # We intersect the grid's samples with the master list
    common_samples = df_orig.index.intersection(master_samples)
    
    # C. Calculate Row Sums (Total Abundance per Sample)
    # axis=1 means sum across columns (species) for each row (sample)
    vol_orig_vec = df_orig.loc[common_samples].sum(axis=1)
    vol_rob_vec  = df_rob.loc[common_samples].sum(axis=1)
    
    # D. Fill the Master Dataframes
    # We use .loc to place the values into the correct rows
    df_vol_orig.loc[common_samples, gid] = vol_orig_vec
    df_vol_rob.loc[common_samples, gid]  = vol_rob_vec
    
    print(f" - Processed Volume for: {gid}")

# --- 4. Final Cleanup & Verification ---
# Optional: Fill remaining NaNs with 0 if logical (meaning sample had 0 abundance)
# df_vol_orig.fillna(0, inplace=True)
# df_vol_rob.fillna(0, inplace=True)

print("\n--- Summary ---")
print(f"df_vol_orig Shape: {df_vol_orig.shape}")
print(f"df_vol_rob Shape:  {df_vol_rob.shape}")

print("\nSample of Original Volumes (G11, first 5 rows):")
print(df_vol_orig["G11"].head())

print("\nSample of Robust Volumes (G11, first 5 rows):")
print(df_vol_rob["G11"].head())

In [ ]:
df_vol_rob

In [ ]:
df_vol_orig

In [ ]:
filtered_species_profile_466

# Volume Reduction Analysis

In [ ]:
import pandas as pd
import numpy as np

# --- 1. Load Data (if not in memory) ---
# Assuming these DataFrames exist from the previous step:
# - filtered_species_profile_466
# - df_vol_orig
# - df_vol_rob

# --- 2. Calculate Total Global Volumes ---
# Sum of every single number in the dataframe
vol_raw  = filtered_species_profile_466.sum().sum()
vol_orig = df_vol_orig.sum().sum()
vol_rob  = df_vol_rob.sum().sum()

print(f"Total Volume (Raw Profile):   {vol_raw:,.2f}")
print(f"Total Volume (Grid Original): {vol_orig:,.2f}")
print(f"Total Volume (Grid Robust):   {vol_rob:,.2f}")

# --- 3. Calculate Reductions ---

# A. Loss due to Gridding (Raw -> Grid Original)
# If this is high (>5%), it means many species were excluded from the grids.
loss_gridding = (vol_raw - vol_orig)
pct_loss_grid = (loss_gridding / vol_raw) * 100

# B. Noise Removal (Grid Original -> Grid Robust)
# This is the "Denoising Power" of your RPCA.
loss_noise = (vol_orig - vol_rob)
pct_loss_noise = (loss_noise / vol_orig) * 100

# C. Total Reduction (Raw -> Robust)
total_loss = (vol_raw - vol_rob)
pct_total_loss = (total_loss / vol_raw) * 100

# --- 4. Print Report ---
print("\n=== VOLUME REDUCTION REPORT ===")
print(f"1. Loss due to Gridding:    {pct_loss_grid:.2f}%")
print(f"   (This confirms if your 16 grids cover most of the species)")

print(f"2. Noise Removed by RPCA:   {pct_loss_noise:.2f}%")
print(f"   (This is the % of data identified as 'Sparse Outliers' or 'Noise')")

print(f"3. Final Clean Signal retained: {100 - pct_total_loss:.2f}%")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# --- 1. Calculate Per-Grid Noise Stats ---
# We use your existing DataFrames
grid_totals_orig = df_vol_orig.sum(axis=0)
grid_totals_rob = df_vol_rob.sum(axis=0)

# Calculate % Noise per grid
grid_noise_pct = ((grid_totals_orig - grid_totals_rob) / grid_totals_orig) * 100
df_grid_stats = pd.DataFrame({
    'Grid_ID': grid_noise_pct.index,
    'Noise_Pct': grid_noise_pct.values,
    'Total_Volume': grid_totals_orig.values
}).sort_values('Grid_ID')

# --- 2. Create the Plots ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot A: Total Volume Reduction (The "Big Picture")
stages = ['Raw Data', 'Grid Original', 'Grid Robust']
volumes = [66422.05, 66422.05, 34143.00] # From your output
colors = ['gray', 'blue', 'green']

bars = axes[0].bar(stages, volumes, color=colors, alpha=0.8, edgecolor='black')
axes[0].set_title("Volum Retention", fontsize=14, fontweight='bold')
axes[0].set_ylabel("Total Abundance Volume")
axes[0].grid(axis='y', alpha=0.3)

# Add Annotation Arrow
axes[0].annotate(
    '-48.6% Noise', 
    xy=(2, 34143), xytext=(1, 66422),
    arrowprops=dict(facecolor='red', shrink=0.05),
    fontsize=12, color='red', fontweight='bold', ha='center'
)

# Plot B: Noise vs Stability per Grid (The "Granular View")
# We stack Stable (Green) on top of Noise (Red) to show the ratio
x = df_grid_stats['Grid_ID']
y_total = df_grid_stats['Total_Volume']
y_noise = y_total * (df_grid_stats['Noise_Pct'] / 100)
y_stable = y_total - y_noise

axes[1].bar(x, y_stable, label='Stable Signal (L)', color='green', alpha=0.7, edgecolor='black')
axes[1].bar(x, y_noise, bottom=y_stable, label='Noise/Outliers (S)', color='red', alpha=0.6, edgecolor='black')

axes[1].set_title("Stability vs Noise per Grid", fontsize=14, fontweight='bold')
axes[1].set_ylabel("Total Abundance")
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# --- 3. Print the "Noisiest" Grids ---
print("--- Grids with Highest Noise Content ---")
print(df_grid_stats.sort_values("Noise_Pct", ascending=False).head())

In [ ]:
import sys

def print_size(name, obj):
    size_bytes = sys.getsizeof(obj)
    size_mb = size_bytes / (1024 ** 2)
    size_gb = size_bytes / (1024 ** 3)
    print(f"{name:<30} : {size_mb:>10.2f} MB  |  {size_gb:>10.2f} GB")

print("-" * 60)
print(f"{'Variable Name':<30} : {'Size (MB)':>10}      {'Size (GB)':>10}")
print("-" * 60)

# Pass your variables here
print_size("Gut_SpeciesProfile", Gut_SpeciesProfile)
print_size("filtered_species_profile_466", filtered_species_profile_466)
print_size("df_vol_rob", df_vol_rob)
print("-" * 60)

In [ ]:
import matplotlib.pyplot as plt
import sys
import pandas as pd

# 1. Define your variables
data_objects = [
    ("Gut_SpeciesProfile", Gut_SpeciesProfile),
    ("filtered_species_profile_466", filtered_species_profile_466),
    ("df_vol_rob", df_vol_rob)
]

# 2. Calculate sizes
names = []
sizes_mb = []
sizes_gb = []

for name, obj in data_objects:
    # Use obj.memory_usage(deep=True).sum() if using real DataFrames
    size_bytes = sys.getsizeof(obj)
    
    names.append(name)
    sizes_mb.append(size_bytes / (1024 ** 2))
    sizes_gb.append(size_bytes / (1024 ** 3))

# 3. Create Plot
plt.figure(figsize=(10, 6))
bars = plt.bar(names, sizes_mb, color=['#1f77b4', '#ff7f0e', '#2ca02c'])

# --- FIX: Add headroom to prevent text cropping ---
# Get the tallest bar height and add 20% extra space on top
max_height = max(sizes_mb)
plt.ylim(0, max_height * 1.2) 
# --------------------------------------------------

plt.ylabel('Memory Usage (MB)')
plt.title('Memory Consumption Comparison')
plt.grid(axis='y', linestyle='--', alpha=0.7)

# 4. Add Labels
for bar, gb_val in zip(bars, sizes_gb):
    height = bar.get_height()
    label_text = f'{height:,.0f} MB\n({gb_val:.2f} GB)'
    
    plt.text(bar.get_x() + bar.get_width()/2., height + (max_height * 0.02),
             label_text,
             ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

# Grid Coherence Analysis (Variance Explained by PC1)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
import os

print("--- Analysis: Grid Coherence (Variance Explained by PC1) ---")

# =========================================================
# 1. SETUP: REDEFINE GRIDS
# =========================================================
# We need to ensure we know which species are in which grid.
# We use the correlation matrix of the 466 species to define the layout.
if 'filtered_species_profile_466' not in globals():
    raise ValueError("⚠️ Data missing. Please ensure 'filtered_species_profile_466' is loaded.")

print("1. Defining Grids based on Species Correlation...")
species_corr = filtered_species_profile_466.corr()

# Run PCA on Species (to find their 'location' in the map)
pca_sp = PCA(n_components=2)
sp_coords = pca_sp.fit_transform(species_corr)
sp_grid_df = pd.DataFrame(sp_coords, columns=['x', 'y'], index=species_corr.index)

# Define 16 Grids (Quantiles)
sp_grid_df['xb'] = pd.qcut(sp_grid_df['x'], 4, labels=[0,1,2,3])
sp_grid_df['yb'] = pd.qcut(sp_grid_df['y'], 4, labels=[0,1,2,3])
sp_grid_df['gid'] = sp_grid_df['xb'].astype(str) + "-" + sp_grid_df['yb'].astype(str)

print("   ✅ Grid definitions ready.")

# =========================================================
# 2. CALCULATE VARIANCE EXPLAINED PER GRID
# =========================================================
grid_stats = []
unique_grids = sorted(sp_grid_df['gid'].unique())

print(f"\n2. Analyzing Local Variance in {len(unique_grids)} Grids...")
print(f"{'Grid':<10} | {'Species':<8} | {'PC1 Var Expl (%)':<20} | {'Status'}")
print("-" * 65)

for g in unique_grids:
    # A. Identify species in this grid
    species_in_g = sp_grid_df[sp_grid_df['gid'] == g].index
    
    # B. Extract their data (Patients x Local_Species)
    # We filter to ensure they exist in our patient dataset
    valid_species = [s for s in species_in_g if s in filtered_species_profile_466.columns]
    
    if len(valid_species) > 1:
        X_local = filtered_species_profile_466[valid_species]
        
        # C. Run Local PCA
        pca_local = PCA(n_components=1)
        pca_local.fit(X_local)
        
        # D. Get Variance Explained Ratio
        var_expl = pca_local.explained_variance_ratio_[0] * 100
        
        status = "High Coherence" if var_expl > 50 else "Low Coherence (Complex)"
        
        grid_stats.append({
            'Grid_ID': g,
            'Species_Count': len(valid_species),
            'PC1_Variance': var_expl
        })
        
        print(f"{g:<10} | {len(valid_species):<8} | {var_expl:.2f}%              | {status}")
        
    elif len(valid_species) == 1:
        grid_stats.append({'Grid_ID': g, 'Species_Count': 1, 'PC1_Variance': 100.0})
        print(f"{g:<10} | 1        | 100.00%              | Single Species")
    else:
        print(f"{g:<10} | 0        | N/A")

# =========================================================
# 3. VISUALIZATION (Heatmap)
# =========================================================
stats_df = pd.DataFrame(grid_stats)
stats_df = stats_df.dropna()

# Map back to X/Y coordinates
stats_df['X'] = stats_df['Grid_ID'].apply(lambda x: int(x.split('-')[0]))
stats_df['Y'] = stats_df['Grid_ID'].apply(lambda x: int(x.split('-')[1]))

# Create Pivot Table
heatmap_data = stats_df.pivot(index='Y', columns='X', values='PC1_Variance')
heatmap_data = heatmap_data.sort_index(ascending=False)

plt.figure(figsize=(10, 8))
sns.heatmap(heatmap_data, annot=True, fmt=".1f", cmap="magma", 
            cbar_kws={'label': '% Variance Explained by PC1'}, vmin=0, vmax=100)

plt.title("Why Compression Failed:\nLow Coherence in Complex Grids", fontsize=16, fontweight='bold', pad=20)
plt.xlabel("X Quantile (Embedding PC1)", fontsize=12)
plt.ylabel("Y Quantile (Embedding PC2)", fontsize=12)
plt.tight_layout()
plt.show()

# Summary
avg_var = stats_df['PC1_Variance'].mean()
print(f"\nAverage Variance Explained by PC1: {avg_var:.2f}%")
if avg_var < 50:
    print("CONCLUSION: The grids are internally complex. A single sum cannot represent them.")
else:
    print("CONCLUSION: The grids are coherent.")

# Generating the QR Code

In [ ]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
import json
import qrcode
import matplotlib.pyplot as plt

In [ ]:


# --- 1. Select the Target Patient ---
# We will use the first patient from your previous outputs as the test case
target_patient = "001_VA"  

# --- 2. Extract PC1 Scores for the Patient ---
patient_signature = {"Patient_ID": target_patient}

print(f"Extracting 16-Grid PC1 Signature for patient: {target_patient}...")

# Loop through the 16 grids in the RPCA results
for gid in sorted(RPCA_Grid_Results.keys()):
    res = RPCA_Grid_Results[gid]
    
    # Ensure the grid processed successfully
    if "L_Matrix" not in res:
        continue
        
    df_robust = res["L_Matrix"]
    
    # Check if patient exists in this grid's data
    if target_patient not in df_robust.index:
        print(f" [!] Patient {target_patient} not found in {gid}")
        patient_signature[gid] = None
        continue
        
    # Fit PCA on the entire Robust L-Matrix to establish the global PC1 axis
    pca = PCA(n_components=1)
    pc1_all = pca.fit_transform(df_robust).flatten()
    
    # Create a Series mapped to the patient IDs
    pc1_series = pd.Series(pc1_all, index=df_robust.index)
    
    # Extract the patient's specific coordinate and round it to 4 decimals
    # (Rounding saves physical space inside the QR code pattern)
    patient_pc1 = round(float(pc1_series[target_patient]), 4)
    patient_signature[gid] = patient_pc1

print("\n--- Extracted Signature ---")
# Displaying formatted JSON for the terminal output
print(json.dumps(patient_signature, indent=2))

# --- 3. Generate the QR Code ---
print("\nGenerating QR Code...")

# Create QR Code object
qr = qrcode.QRCode(
    version=1, # Auto-scales based on data size
    error_correction=qrcode.constants.ERROR_CORRECT_L, # Low error correction keeps the QR code simple/clean
    box_size=10,
    border=4,
)

# We encode a highly compact JSON string to keep the QR code easy to scan
compact_json = json.dumps(patient_signature) 
qr.add_data(compact_json)
qr.make(fit=True)

# Create the image
img = qr.make_image(fill_color="black", back_color="white")

# Save the image to your directory
output_filename = f"Microbiome_QR_{target_patient}.png"
img.save(output_filename)
print(f"[Success] QR Code saved to: {output_filename}")

# --- 4. Display the QR Code inline ---
plt.figure(figsize=(6, 6))
plt.imshow(img, cmap='gray')
plt.axis('off')
plt.title(f"Microbiome Signature QR:\nSample Id : {target_patient}", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# After running qr.make(fit=True) in your previous script:
version_used = qr.version
matrix_size = (version_used * 4) + 17

print(f"Data Length: {len(compact_json)} characters/bytes")
print(f"QR Code Version Auto-Selected: {version_used}")
print(f"Physical Matrix Size: {matrix_size}x{matrix_size} modules")

# Build a "Master Signature" Database for 70,185 samples

In [ ]:
import pandas as pd
from sklearn.decomposition import PCA
import numpy as np

print("Building Master Signature Database for all 70,185 samples...")

signature_dict = {}

# Loop through all 16 grids
for gid in sorted(RPCA_Grid_Results.keys()):
    res = RPCA_Grid_Results[gid]
    
    if "L_Matrix" not in res:
        continue
        
    df_robust = res["L_Matrix"]
    
    # Fit PCA on the entire Robust L-Matrix
    pca = PCA(n_components=1)
    pc1_all = pca.fit_transform(df_robust).flatten()
    
    # Store the results in a Series, rounded to 4 decimals to save space
    signature_dict[gid] = pd.Series(pc1_all, index=df_robust.index).round(4)

# Combine all 16 Series into a single Master DataFrame
df_master_signatures = pd.DataFrame(signature_dict)

# Save to CSV
output_csv = "Microbiome_Master_Signatures.csv"
df_master_signatures.to_csv(output_csv)

print(f"[Success] Master Database saved to {output_csv}")
print(f"Shape: {df_master_signatures.shape}")
print(df_master_signatures.head())

In [ ]:
import pandas as pd
import json
import qrcode
import matplotlib.pyplot as plt
import numpy as np

# --- 1. Load the Master Database ---
# We load this once so the function can run instantly when called
csv_path = "Microbiome_Master_Signatures.csv"
try:
    # Assuming the first column (rownames) is the Patient ID
    df_master = pd.read_csv(csv_path, index_col=0)
    print(f"✅ Master Database loaded successfully! Shape: {df_master.shape}")
except FileNotFoundError:
    print(f"❌ Error: Could not find {csv_path}. Make sure it is in your current directory.")

# --- 2. Define the On-Demand Function ---
def show_microbiome_qr(patient_id, df=df_master):
    """
    Instantly generates and displays the microbiome state QR code for a given patient.
    Does NOT save the image to disk.
    """
    # Check if patient exists in the database
    if patient_id not in df.index:
        print(f"⚠️ Patient ID '{patient_id}' not found in the database.")
        return
    
    print(f"Extracting microbiome state for: {patient_id}...")
    
    # Extract the 16 grid values
    patient_data = df.loc[patient_id].to_dict()
    
    # Build the final dictionary
    signature_dict = {"Patient_ID": patient_id}
    signature_dict.update(patient_data)
    
    # Convert to a compact JSON string (no spaces to save QR capacity)
    compact_json = json.dumps(signature_dict, separators=(',', ':'))
    
    # Generate the QR Code in memory
    qr = qrcode.QRCode(
        version=1, # Will auto-scale as needed
        error_correction=qrcode.constants.ERROR_CORRECT_L,
        box_size=10,
        border=4,
    )
    qr.add_data(compact_json)
    qr.make(fit=True)
    
    # Create the image object
    img = qr.make_image(fill_color="black", back_color="white")
    
    # Convert to numpy array for Matplotlib display
    img_array = np.array(img.convert('RGB'))
    
    # Display the image on screen
    plt.figure(figsize=(6, 6))
    plt.imshow(img_array)
    plt.axis('off') # Hide the axes
    plt.title(f"Microbiome State QR\nSample Id: {patient_id}", fontsize=14, fontweight='bold', pad=15)
    plt.tight_layout()
    plt.show()

# --- 3. Test It Out ---
# You can change this to any patient ID in your dataset!
show_microbiome_qr("004_RF")

In [ ]:
import sys
sys.getsizeof("Microbiome_Master_Signatures.csv")

In [ ]:
import pandas as pd
import qrcode
import matplotlib.pyplot as plt
import numpy as np
import io
import sys

# --- 1. Load the Master Database ---
csv_path = "Microbiome_Master_Signatures.csv"

try:
    df_master = pd.read_csv(csv_path, index_col=0)
    print(f"✅ Master Database loaded successfully! Shape: {df_master.shape}")
except FileNotFoundError:
    raise FileNotFoundError("❌ Microbiome_Master_Signatures.csv not found")


# --- 2. QR Generator Function ---
def show_microbiome_qr(patient_id, df=df_master):
    """
    Generates and displays a QR code that links to the hosted microbiome report.
    """

    # Validate patient ID
    if patient_id not in df.index:
        print(f"⚠️ Patient ID '{patient_id}' not found in database")
        return

    print(f"Extracting microbiome state for: {patient_id}")
    
# --- IMPORTANT ---
    # QR contains a URL that points to your NEW GitHub page
    url = f"https://debarshiroy-056.github.io/Image_Based_Analysis_Microbiome/sample.html?id={patient_id}"

    # --- QR Generation ---
    qr = qrcode.QRCode(
        version=None,  # Auto-select QR size
        error_correction=qrcode.constants.ERROR_CORRECT_L,
        box_size=10,
        border=4,
    )

    qr.add_data(url)
    qr.make(fit=True)

    img = qr.make_image(fill_color="black", back_color="white")

    # Convert to numpy array for matplotlib
    img_array = np.array(img.convert("RGB"))

    # Check QR code size in bytes
    img_bytes_io = io.BytesIO()
    img.save(img_bytes_io, format="PNG")
    size_bytes = img_bytes_io.tell()
    img_bytes_io.seek(0)
    print(f"📏 QR Code Size: {size_bytes} bytes")
    print(f"📊 QR Code Version: {qr.version}, Box Size: {qr.box_size}, Border: {qr.border}")

    # --- Display QR ---
    plt.figure(figsize=(6,6))
    plt.imshow(img_array)
    plt.axis("off")
    plt.title(
        f"Microbiome State QR\nSample ID: {patient_id}",
        fontsize=14,
        fontweight="bold",
        pad=15
    )
    plt.tight_layout()
    plt.show()

    print(f"\n🔗 QR Encodes URL:\n{url}")


# --- 3. Test Example ---
show_microbiome_qr("004_RF")